# Congressional Bill Actions and Senator Mentions using Regular Expressions
### DATASCI 530 · Assignment 4 · Whitney Barr
### Data source: Bills_actions.csv from Lecture 9

# Summary of Findings 
- 1,494 distinct bills in the 116th Congressional term
- Amendments are the largest category at 46% of rows. House bills come next at 27%.
- House floor actions are the most common main action at 31%.
- 116 House floor actions involve Senate bills.
- Takano is mentioned on the most distinct bills (9), followed by Dingell and Haaland (6 each).


In [75]:
## Setup
#import packages

import pandas as pd
bills_actions = pd.read_csv(r"C:\Users\Owner\Documents\GitHub\datasci530fall2026\Lecture 09\data_raw\bills_actions.csv")

# 1. How many distinct “bill” identifiers are there? What  proportion  of  rows  belongs  to  each  category?  What proportion belongs to different “main_action”? 


In [76]:
#A bill is identified by Congress + bill_type + bill_number
bills_ids = bills_actions.groupby(["Congress", "bill_type", "bill_number"])

In [77]:
print("Distinct bills:", len(bills_ids.size()))

Distinct bills: 1494


There are **1494** distinct bills. Each bill identifies the specific two year Congressional term in which the bill was introduced, type of bill (such as House or Senate-related bill), and the number of the bill within the Congressional term.

In [89]:
category_counts = bills_actions["category"].value_counts()

category_table = pd.DataFrame({
    "Action rows": category_counts,
    "Percent of rows": category_counts / len(bills_actions) * 100
})
category_table["Percent of rows"] = category_table["Percent of rows"].round(2)

(category_table.style
    .format({"Action rows": "{:,}", "Percent of rows": "{:.2f}%"})
    .set_caption("Distribution of bill categories"))
#this code is used to determine #/% of row data. How many of each distinct row exist to the column (in this case category? {percent:.2f} inserts the percentage rounded to two decimal places (2 digits after the decimal)

,Action rows,Percent of rows
category,,
amendment,"1,529",46.29%
house bill,902,27.31%
senate bill,514,15.56%
house resolution,234,7.08%
senate resolution,60,1.82%
house joint resolution,22,0.67%
house concurrent resolution,20,0.61%
senate concurrent resolution,14,0.42%
senate joint resolution,8,0.24%


In [79]:
main_action_counts = bills_actions["main_action"].value_counts()

# Print the raw count and percentage for each main action
print("Counts and percentages by main_action:")
for main_action in main_action_counts.index:
    count = main_action_counts[main_action]
    percent = count / len(bills_actions) * 100
    print(f"{main_action}: {count} ({percent:.2f}%)")

# Put the same results in a table, sorted from most rows to fewest
main_action_table = pd.DataFrame({
    "Count": main_action_counts,
    "Percent": main_action_counts / len(bills_actions) * 100
})
main_action_table["Percent"] = main_action_table["Percent"].round(2)
main_action_table = main_action_table.sort_values("Count", ascending=False)
main_action_table = main_action_table.rename(columns={
    "Count": "Action rows",
    "Percent": "Percent of rows"
})
main_action_table.index.name = "Main action"

(main_action_table.style
    .format({"Percent of rows": "{:.2f}%"})
    .set_caption("Distribution of congressional actions"))

Counts and percentages by main_action:
house floor actions: 1020 (30.88%)
house amendment offered: 828 (25.07%)
senate committee/subcommittee actions: 434 (13.14%)
other house amendment actions: 397 (12.02%)
senate amendment proposed (on the floor): 286 (8.66%)
house committee/subcommittee actions: 218 (6.60%)
reported to senate: 36 (1.09%)
senate committee report filed after reporting: 33 (1.00%)
other senate amendment actions: 18 (0.54%)
resolving differences -- house actions: 14 (0.42%)
senate floor actions: 13 (0.39%)
house discharge petition filed: 3 (0.09%)
resolving differences -- senate actions: 3 (0.09%)


,Action rows,Percent of rows
Main action,,
house floor actions,1020,30.88%
house amendment offered,828,25.07%
senate committee/subcommittee actions,434,13.14%
other house amendment actions,397,12.02%
senate amendment proposed (on the floor),286,8.66%
house committee/subcommittee actions,218,6.60%
reported to senate,36,1.09%
senate committee report filed after reporting,33,1.00%
other senate amendment actions,18,0.54%


# 2. How many House floor actions are associated with bills in the “Senate bills” category? 

- What proportion of these actions were suspended? What proportion of these 
actions were suspended as amended? 

- The  House  floor  actions  mention  specific  people  by  (Mr.,  Ms.,  Mrs.,  etc).  Use 
regular  expressions  to  extract  the  names  of  specific  people.  Produce  a  new 
dataset counting how many bills mention the same person and then extract the 
unique list of names of the people mentioned. What is the distribution of total 
bills proposed per person?

In [80]:
# Select House floor actions associated with Senate bills
house_senate_actions = bills_actions[
    (bills_actions["main_action"] == "house floor actions")
    & (bills_actions["category"] == "senate bill")
].copy()

print("House floor actions on Senate bills:", len(house_senate_actions))

# Count suspension actions and suspension actions marked as amended
suspended_actions = house_senate_actions[
    house_senate_actions["action"].str.contains("suspend")
]
suspended_amended = suspended_actions[
    suspended_actions["action"].str.contains("as amended")
]

print("Suspended:", len(suspended_actions),
      "(", round(len(suspended_actions) / len(house_senate_actions) * 100, 2), "%)")
print("Suspended as amended:", len(suspended_amended),
      "(", round(len(suspended_amended) / len(house_senate_actions) * 100, 2), "% )")

# Find all people mentioned in each action
people_by_action = house_senate_actions["action"].str.findall(
    "Mr\\. [A-Za-z'-]+|Ms\\. [A-Za-z'-]+|Mrs\\. [A-Za-z'-]+"
)

# Count each person once per distinct bill
person_bills = {}
for i in range(len(house_senate_actions)):
    bill_id = (house_senate_actions["Congress"].iloc[i],
               house_senate_actions["bill_type"].iloc[i],
               house_senate_actions["bill_number"].iloc[i])
    for person in people_by_action.iloc[i]:
        if person not in person_bills:
            person_bills[person] = []
        if bill_id not in person_bills[person]:
            person_bills[person].append(bill_id)

person_rows = []
for person in person_bills:
    person_rows.append({"Person": person,
                        "Bills mentioned": len(person_bills[person])})

person_bill_table = pd.DataFrame(person_rows)
person_bill_table = person_bill_table.sort_values(
    "Bills mentioned", ascending=False
).reset_index(drop=True)
person_bill_table.index = range(1, len(person_bill_table) + 1)
person_bill_table.index.name = "Rank"

(person_bill_table.style
    .format({"Bills mentioned": "{:,}"})
    .set_caption("Bills Mentioned by House Member"))

House floor actions on Senate bills: 116
Suspended: 80 ( 68.97 %)
Suspended as amended: 18 ( 15.52 % )


,Person,Bills mentioned
Rank,,
1,Mr. Takano,9
2,Mrs. Dingell,6
3,Ms. Haaland,6
4,Mr. Case,5
5,Mr. Pallone,5
6,Ms. Bass,5
7,Ms. Kelly,5
8,Mr. Cicilline,4
9,Mr. DeFazio,4


# 3. Now  focus  on  a  subset  of  bills  whose  actions  are  marked  as  “Senate  amendment proposed (on the floor)”. An important feature of the text is that it can mention one or more senators. 

- Produce a new dataset counting how many senators are mentioned per bill. This 
dataset  should  also  include  two  variables,  with  the  name  of  the  first  senator 
(Senator 1) mentioned in the bill and the name of the second senator (Senator 
2). What is the distribution of the number of senators per bill? 

- With how many different people does each Senator coauthor bills, i.e. how many 
different people appear as Senator 2 for each Senator 1?

- Briefly describe your insights.

In [ ]:
# Select Senate amendment actions from the floor
senate_amendments = bills_actions[
    bills_actions["main_action"] == "senate amendment proposed (on the floor)"
].copy()

# Find senator names in each action
senator_matches = senate_amendments["action"].str.findall(
    "Senator [A-Za-z'-]+"
)

# Gather unique senators mentioned for each bill
bill_senators = {}
for i in range(len(senate_amendments)):
    bill_id = (senate_amendments["Congress"].iloc[i],
               senate_amendments["bill_type"].iloc[i],
               senate_amendments["bill_number"].iloc[i])
    if bill_id not in bill_senators:
        bill_senators[bill_id] = []
    for senator in senator_matches.iloc[i]:
        if senator not in bill_senators[bill_id]:
            bill_senators[bill_id].append(senator)

# Create one row per bill with senator count and the first two names
senator_rows = []
for bill_id in bill_senators:
    names = bill_senators[bill_id]
    if len(names) > 0:
        senator_1 = names[0]
    else:
        senator_1 = "Not listed"
    if len(names) > 1:
        senator_2 = names[1]
    else:
        senator_2 = "Not listed"

    senator_rows.append({"Congress": bill_id[0],
                         "Bill type": bill_id[1],
                         "Bill number": bill_id[2],
                         "Number of senators": len(names),
                         "Senator 1": senator_1,
                         "Senator 2": senator_2})

senator_bills = pd.DataFrame(senator_rows)
senator_counts = senator_bills["Number of senators"].value_counts().sort_index()
print("Distinct Senate bills:", len(senator_bills))

# Count each Senator 2 once for each Senator 1
coauthors = {}
for i in range(len(senator_bills)):
    senator_1 = senator_bills["Senator 1"].iloc[i]
    senator_2 = senator_bills["Senator 2"].iloc[i]
    if not (senator_1 == "Not listed") and not (senator_2 == "Not listed"):
        if not (senator_1 in coauthors):
            coauthors[senator_1] = []
        if not (senator_2 in coauthors[senator_1]):
            coauthors[senator_1].append(senator_2)

coauthor_rows = []
for senator_1 in coauthors:
    coauthor_rows.append({"Senator 1": senator_1,
                          "Distinct coauthors": len(coauthors[senator_1])})

coauthor_table = pd.DataFrame(coauthor_rows)
coauthor_table = coauthor_table.sort_values(
    "Distinct coauthors", ascending=False
).reset_index(drop=True)

Distinct Senate bills: 131


In [90]:
senator_bills.head(10).style.hide(axis="index").set_caption(
    "Sample Senate Bills and Senator Mentions"
)

Congress,Bill type,Bill number,Number of senators,Senator 1,Senator 2
116,s,1029,2,Senator McConnell,Senator Cornyn
116,s,1031,1,Senator Hawley,Not listed
116,s,1160,2,Senator McConnell,Senator Smith
116,s,1253,2,Senator Feinstein,Senator Cornyn
116,s,1309,2,Senator McConnell,Senator Cardin
116,s,1340,2,Senator Rubio,Senator Menendez
116,s,1387,2,Senator Moran,Senator Braun
116,s,1434,2,Senator McConnell,Senator Portman
116,s,1430,2,Senator Fischer,Senator Lankford
116,s,1636,2,Senator McConnell,Senator Alexander


In [91]:
senator_distribution_table = pd.DataFrame({
    "Bill count": senator_counts,
    "Percent of bills": senator_counts / len(senator_bills) * 100
})
senator_distribution_table["Percent of bills"] = senator_distribution_table["Percent of bills"].round(2)
senator_distribution_table.index.name = "Number of senators"

(senator_count_table.style
    .set_caption("Top 10 Senators Mentioned in Bill Actions")
    .apply(
        lambda row: ["background-color: hotpink; color: black"] * len(row)
        if row.name <= 3 else [""] * len(row),
        axis=1
    ))

,Senator,Mentions
Rank,,
1,Senator McConnell,179
2,Senator Johnson,143
3,Senator Murkowski,98
4,Senator Risch,76
5,Senator Barrasso,42
6,Senator Hoeven,42
7,Senator Wicker,36
8,Senator Moran,33
9,Senator Reed,32


In [84]:
coauthor_table.index = range(1, len(coauthor_table) + 1)
coauthor_table.index.name = "Rank"

(coauthor_table.style
    .format({"Distinct coauthors": "{:,}"})
    .set_caption("Distinct Senator 2 coauthors by Senator 1"))

,Senator 1,Distinct coauthors
Rank,,
1,Senator McConnell,38
2,Senator Cornyn,9
3,Senator Murkowski,6
4,Senator Braun,5
5,Senator Moran,4
6,Senator Sasse,3
7,Senator Paul,3
8,Senator Thune,3
9,Senator Rubio,2


**Most bills (99 of 131, or 75.57%) mention two senators.** McConnell appears as Senator 1 with 38 distinct Senator 2 coauthors, compared with Cornyn’s 9.

# 4. Compute an additional table/figure not included in the questions above. What do we learn from it?

The table below shows that **Senator McConnell** is mentioned most frequently across all bill actions **(179 times)**, followed by **Senator Johnson (143)** and **Senator Murkowski (98)**, suggesting mentions are concentrated amongst a few senators.

In [88]:
senator_mentions = bills_actions["action"].str.findall(
    "Senator [A-Za-z'-]+"
)

senator_mention_counts = {}
for names in senator_mentions:
    for name in names:
        if name not in senator_mention_counts:
            senator_mention_counts[name] = 0
        senator_mention_counts[name] += 1

senator_count_rows = []
for name in senator_mention_counts:
    senator_count_rows.append({
        "Senator": name,
        "Mentions": senator_mention_counts[name]
    })
senator_count_table = pd.DataFrame(senator_count_rows)
senator_count_table = senator_count_table.sort_values(
    "Mentions", ascending=False
).head(10).reset_index(drop=True)

senator_count_table.index = senator_count_table.index + 1
senator_count_table.index.name = "Rank"
senator_count_table

,Senator,Mentions
Rank,,
1,Senator McConnell,179
2,Senator Johnson,143
3,Senator Murkowski,98
4,Senator Risch,76
5,Senator Barrasso,42
6,Senator Hoeven,42
7,Senator Wicker,36
8,Senator Moran,33
9,Senator Reed,32
